# User-Based Collaborative Filtering

## Objective

In this notebook, we implement a User-Based Collaborative Filtering recommendation system.

Unlike popularity-based recommendation, collaborative filtering provides personalized recommendations by identifying users with similar rating patterns.

Workflow:

- Load processed dataset
- Create User–Movie Interaction Matrix
- Compute User Similarity
- Find Similar Users
- Generate Personalized Recommendations
- Evaluate the recommendation quality

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
movie_ratings = pd.read_csv(
    "processed_data/movie_ratings_processed.csv"
)

movie_ratings.head()

,user_id,movie_id,rating,timestamp,movie_title,user_encoded,movie_encoded
0,196,242,3,881250949,Kolya (1996),195,241
1,186,302,3,891717742,L.A. Confidential (1997),185,301
2,22,377,1,878887116,Heavyweights (1994),21,376
3,244,51,2,880606923,Legends of the Fall (1994),243,50
4,166,346,1,886397596,Jackie Brown (1997),165,345


In [3]:
user_movie_matrix = movie_ratings.pivot_table(
    index="user_id",
    columns="movie_title",
    values="rating"
)

user_movie_matrix.head()

movie_title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,5.0,3.0,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,2.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,4.0,NaN


In [4]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)

user_movie_matrix_filled.head()

movie_title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,2.0,5.0,0.0,0.0,3.0,4.0,0.0,0.0,...,0.0,0.0,0.0,5.0,3.0,0.0,0.0,0.0,4.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,2.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,4.0,0.0


## Why Do We Replace `NaN` with `0` Here?

When creating the **User–Movie Interaction Matrix**, many entries are missing because users have not rated every movie.

For example:

| User | Movie A | Movie B | Movie C |
|------|---------|---------|---------|
| User 1 | 5 | NaN | 4 |
| User 2 | 4 | 3 | NaN |

Here:

- `NaN` means **the user has never rated that movie**.
- It **does not** mean the user disliked the movie or gave it a rating of zero.

### Why not keep `NaN`?

The **Cosine Similarity** algorithm cannot perform mathematical operations on missing (`NaN`) values.

Therefore, before computing similarity, we replace missing values with **0**.

```python
user_movie_matrix_filled = user_movie_matrix.fillna(0)
```

### Does `0` Mean the User Rated the Movie Zero?

**No.**

In this context:

- `0` simply represents **"No interaction recorded."**
- It is **not** treated as an actual movie rating.

### Why Is This Correct?

This notebook implements **Memory-Based Collaborative Filtering**, where we calculate similarity between users using their rating vectors.

Replacing `NaN` with `0` allows the cosine similarity algorithm to compare users while interpreting missing ratings as the absence of interaction.

### Important Note

For **Deep Learning Recommendation Models** (such as Embedding Models in PyTorch), we **do not** replace missing values with zero.

Instead, we train directly using the original interaction records:

- User ID
- Movie ID
- Rating

The interaction matrix with zeros is used only for similarity-based recommendation methods.

### Summary

- `NaN` → User has never rated the movie.
- `0` → Placeholder representing no interaction.
- Used only for cosine similarity calculations.
- Not used when training deep learning recommendation models.

In [5]:
user_similarity = cosine_similarity(user_movie_matrix_filled)

user_similarity.shape

(943, 943)

In [6]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix_filled.index,
    columns=user_movie_matrix_filled.index
)

user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.168937,0.048388,0.064561,0.379670,0.429682,0.443097,0.320079,0.078385,0.377733,...,0.372213,0.119860,0.269860,0.193343,0.197949,0.118722,0.315064,0.149086,0.181612,0.399432
2,0.168937,1.000000,0.113393,0.179694,0.073623,0.242106,0.108604,0.104257,0.162470,0.161273,...,0.147095,0.310661,0.363328,0.410725,0.322713,0.231096,0.228793,0.162911,0.175273,0.106732
3,0.048388,0.113393,1.000000,0.349781,0.021592,0.074018,0.067423,0.084419,0.062039,0.066217,...,0.033885,0.043453,0.167140,0.071288,0.126278,0.026758,0.164539,0.102899,0.136757,0.026990
4,0.064561,0.179694,0.349781,1.000000,0.031804,0.068431,0.091507,0.188060,0.101284,0.060859,...,0.054615,0.036784,0.133619,0.196561,0.146058,0.030202,0.196858,0.152041,0.171538,0.058752
5,0.379670,0.073623,0.021592,0.031804,1.000000,0.238636,0.374733,0.248930,0.056847,0.201427,...,0.340183,0.080580,0.095284,0.081053,0.148607,0.071612,0.239955,0.139595,0.153799,0.313941


## Step 5: Computing User Similarity

The core idea of **User-Based Collaborative Filtering** is to identify users with similar preferences.

To measure similarity between users, we use **Cosine Similarity**.

### What is Cosine Similarity?

Cosine Similarity measures the similarity between two users by comparing their rating vectors.

Its value ranges from **0 to 1**:

- **1.0** → Users have almost identical preferences.
- **0.0** → Users have no similarity in their rating patterns.

### Why Cosine Similarity?

Cosine Similarity focuses on the direction of rating vectors rather than their magnitude.

For example:

User A:
```
[5, 4, 0, 3]
```

User B:
```
[4, 5, 0, 3]
```

Although the ratings are not exactly the same, both users have similar preferences, resulting in a high cosine similarity score.

### Summary

- Input: User–Movie Interaction Matrix
- Output: User Similarity Matrix
- Algorithm: Cosine Similarity
- Purpose: Find users with similar movie preferences

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix_filled)

print("User Similarity Matrix Shape:")
print(user_similarity.shape)

User Similarity Matrix Shape:
(943, 943)


## Step 6: Convert Similarity Matrix to DataFrame

The cosine similarity function returns a NumPy array.

To make the similarity scores easier to read and access, we convert it into a Pandas DataFrame using the original User IDs as both row and column labels.

In [9]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix_filled.index,
    columns=user_movie_matrix_filled.index
)

user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.168937,0.048388,0.064561,0.379670,0.429682,0.443097,0.320079,0.078385,0.377733,...,0.372213,0.119860,0.269860,0.193343,0.197949,0.118722,0.315064,0.149086,0.181612,0.399432
2,0.168937,1.000000,0.113393,0.179694,0.073623,0.242106,0.108604,0.104257,0.162470,0.161273,...,0.147095,0.310661,0.363328,0.410725,0.322713,0.231096,0.228793,0.162911,0.175273,0.106732
3,0.048388,0.113393,1.000000,0.349781,0.021592,0.074018,0.067423,0.084419,0.062039,0.066217,...,0.033885,0.043453,0.167140,0.071288,0.126278,0.026758,0.164539,0.102899,0.136757,0.026990
4,0.064561,0.179694,0.349781,1.000000,0.031804,0.068431,0.091507,0.188060,0.101284,0.060859,...,0.054615,0.036784,0.133619,0.196561,0.146058,0.030202,0.196858,0.152041,0.171538,0.058752
5,0.379670,0.073623,0.021592,0.031804,1.000000,0.238636,0.374733,0.248930,0.056847,0.201427,...,0.340183,0.080580,0.095284,0.081053,0.148607,0.071612,0.239955,0.139595,0.153799,0.313941


## Understanding the User Similarity Matrix

Each row and column represents a user.

The value at the intersection of two users represents their similarity score.

For example:

| User | User | Similarity |
|------|------|-----------|
| 1 | 1 | 1.000 |
| 1 | 5 | 0.412 |
| 1 | 20 | 0.083 |

### Interpretation

- A value of **1.0** means the users are identical (every user is perfectly similar to themselves).
- Higher values indicate more similar movie preferences.
- Lower values indicate different preferences.

This similarity matrix will be used to identify the nearest neighbors for each user and generate personalized movie recommendations.

In [ ]:
## Understanding the Similarity Scores

Each value in the User Similarity Matrix represents how similar two users are based on their movie ratings.

For example, considering **User 1**:

| Compared User | Similarity Score |
|---------------|----------------:|
| User 1 | 1.000 |
| User 7 | 0.443 |
| User 6 | 0.430 |
| User 5 | 0.380 |
| User 3 | 0.048 |

### Interpretation

- **1.000** → The same user (perfect similarity).
- **0.443** → Very similar movie preferences.
- **0.380** → Moderately similar preferences.
- **0.048** → Very different movie preferences.

The recommendation system will use these similarity scores to identify users with similar tastes and recommend movies that similar users have enjoyed.

## Step 7: Finding Similar Users

After computing the similarity matrix, the next step is to identify the users who have the most similar movie preferences.

For a given user:

1. Retrieve their similarity scores with all other users.
2. Exclude the user themselves (similarity = 1.0).
3. Sort the remaining users in descending order of similarity.
4. Select the top K most similar users.

These similar users are called **Nearest Neighbors** and are used to generate personalized recommendations.

In [11]:
def get_similar_users(user_id, top_n=10):
    """
    Returns the Top-N users most similar to the given user.
    """

    similar_users = user_similarity_df[user_id].sort_values(
        ascending=False
    )

    # Remove the user themselves
    similar_users = similar_users.drop(user_id)

    return similar_users.head(top_n)

In [12]:
similar_users = get_similar_users(user_id=1)

print(similar_users)

user_id
916    0.566293
864    0.549277
457    0.541364
92     0.539415
268    0.538883
435    0.535961
738    0.528695
429    0.527610
303    0.523387
276    0.523277
Name: 1, dtype: float64


## Why Do We Exclude the Same User?

When computing cosine similarity, every user is perfectly similar to themselves.

For example:

| User | Similarity |
|------|-----------:|
| User 1 | **1.000** |

However, recommending movies based on the same user's ratings does not provide any new information.

Therefore, we remove the target user from the similarity list before selecting the nearest neighbors.

## Step 8: Generating Personalized Movie Recommendations

After identifying users with similar preferences, the next step is to recommend movies.

The recommendation process is:

1. Find the Top-K most similar users.
2. Collect movies that these users have rated highly.
3. Remove movies that the target user has already watched.
4. Rank the remaining movies based on their average rating from similar users.
5. Recommend the Top-N highest ranked movies.

This produces personalized recommendations because different users have different nearest neighbors.

In [13]:
def get_watched_movies(user_id):
    """
    Returns all movies already rated by the target user.
    """

    watched = movie_ratings[
        movie_ratings["user_id"] == user_id
    ]["movie_title"].tolist()

    return watched

In [14]:
watched = get_watched_movies(1)

print("Movies Watched :", len(watched))
print(watched[:10])

Movies Watched : 272
['Three Colors: White (1994)', 'Grand Day Out, A (1992)', 'Desperado (1995)', 'Glengarry Glen Ross (1992)', 'Angels and Insects (1995)', 'Groundhog Day (1993)', 'Delicatessen (1991)', 'Hunt for Red October, The (1990)', 'Dirty Dancing (1987)', 'Rock, The (1996)']


## Why Do We Remove Already Watched Movies?

The goal of a recommendation system is to suggest **new movies**.

If a user has already watched a movie, recommending it again usually provides little value.

Therefore, all previously watched movies are removed before generating recommendations.

In [15]:
def recommend_movies(user_id, top_n=10):

    # Similar users
    similar_users = get_similar_users(user_id, top_n=10)

    # Movies watched by target user
    watched_movies = get_watched_movies(user_id)

    # Ratings from similar users
    similar_users_ratings = movie_ratings[
        movie_ratings["user_id"].isin(similar_users.index)
    ]

    # Remove watched movies
    recommendations = similar_users_ratings[
        ~similar_users_ratings["movie_title"].isin(watched_movies)
    ]

    # Rank movies
    recommendations = recommendations.groupby(
        "movie_title"
    )["rating"].mean()

    recommendations = recommendations.sort_values(
        ascending=False
    )

    return recommendations.head(top_n)

## How Does the Recommendation Algorithm Work?

For a target user:

- Find similar users using cosine similarity.
- Collect movies rated by those users.
- Exclude movies already watched by the target user.
- Compute the average rating for each remaining movie.
- Recommend the highest-rated unseen movies.

This is the core workflow of a memory-based User-Based Collaborative Filtering recommender.

In [16]:
recommend_movies(1)

movie_title
Wings of Desire (1987)                       5.0
American in Paris, An (1951)                 5.0
Little Buddha (1993)                         5.0
Secrets & Lies (1996)                        5.0
Die xue shuang xiong (Killer, The) (1989)    5.0
Sophie's Choice (1982)                       5.0
Some Folks Call It a Sling Blade (1993)      5.0
Stealing Beauty (1996)                       5.0
Diva (1981)                                  5.0
Shadowlands (1993)                           5.0
Name: rating, dtype: float64

### Interview Question

**Q. Why do we use ratings from similar users instead of all users?**

**Answer:**

Users with similar preferences are more likely to enjoy similar movies.

Using ratings from all users would ignore individual preferences and produce generic recommendations.

Collaborative Filtering improves personalization by leveraging the opinions of users with similar tastes.

In [17]:
recommend_movies(50)

movie_title
Angels and Insects (1995)                               5.0
Citizen Ruth (1996)                                     5.0
Close Shave, A (1995)                                   5.0
Cement Garden, The (1993)                               5.0
Chungking Express (1994)                                5.0
Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)    5.0
Mina Tannenbaum (1994)                                  5.0
Portrait of a Lady, The (1996)                          5.0
Kolya (1996)                                            5.0
Ma vie en rose (My Life in Pink) (1997)                 5.0
Name: rating, dtype: float64

In [18]:
recommend_movies(200)

movie_title
Wallace & Gromit: The Best of Aardman Animation (1996)    5.0
William Shakespeare's Romeo and Juliet (1996)             5.0
Arsenic and Old Lace (1944)                               5.0
Tomorrow Never Dies (1997)                                5.0
Alien: Resurrection (1997)                                5.0
Ran (1985)                                                5.0
Lost in Space (1998)                                      5.0
Tank Girl (1995)                                          5.0
Sliver (1993)                                             5.0
Full Monty, The (1997)                                    5.0
Name: rating, dtype: float64

# Conclusion

In this notebook, we successfully implemented a **User-Based Collaborative Filtering** recommendation system.

## Tasks Completed

- Loaded the processed MovieLens 100K dataset.
- Created the User–Movie Interaction Matrix.
- Replaced missing values with zero for similarity computation.
- Computed user similarity using Cosine Similarity.
- Identified the Top-K most similar users.
- Generated personalized movie recommendations.
- Excluded movies already watched by the target user.

## Key Learning

Unlike popularity-based recommendation systems, User-Based Collaborative Filtering provides **personalized recommendations** by leveraging the preferences of users with similar tastes.

This notebook serves as the first personalized recommendation model in our recommendation system pipeline.

## Limitation

The current implementation treats all similar users equally while generating recommendations.

In the next notebook, we will improve the recommendation quality by assigning **higher weights to users with higher similarity scores** using **Weighted User-Based Collaborative Filtering**.